# Hyperliquid consolidated account analytics

Combines balances, account history, fills, funding, positions, and PnL across the configured Hyperliquid wallets. Combined account-value history is shown only over the dates available for every wallet; charts show the oldest wallet's account history before shared coverage begins, then the combined account value.

Daily rollups and calendar-period cutoffs use `America/New_York`. Previously saved UTC-aggregated daily totals are mapped to Eastern date labels when loaded; their original intraday split cannot be reconstructed.


## Wallets and implementation

Edit `WALLET_ADDRESSES` here. Both addresses found in the single-wallet notebook are included: the active address and the previously commented address.


In [1]:
from hl_multi_account_analytics_implementation import MultiWalletAnalytics

WALLET_ADDRESSES = [
    '0xf3fb3598bFfA608d1e7CF2837d74F02B1191c248',
    '0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2',
]

ATR_PERIOD = 14
ATR_STOP_MULTIPLE = 1.25
MARGIN_RESERVE_PCT = 20.0
AGGREGATE_RISK_PCT = 20.0

analytics=MultiWalletAnalytics(WALLET_ADDRESSES,data_dir='data')
analytics.position_atr_period=ATR_PERIOD
analytics.position_atr_stop_multiple=ATR_STOP_MULTIPLE
analytics.margin_reserve_pct=MARGIN_RESERVE_PCT
analytics.aggregate_risk_pct=AGGREGATE_RISK_PCT


## Fetch and consolidate wallet data

Fetch each wallet once and prepare combined account history and fills.


In [2]:
%%capture
_ = analytics.fetch_data()


## Highlights

What went well and what needs attention since this notebook last ran, over the last 24 hours, and right now. PnL includes open positions and excludes deposits and withdrawals. Each run saves a baseline in `data/hl_highlights_state.json`.

In [3]:
analytics.show_highlights()


Limited daily history; ATR uses all completed candles available (shorter than 14 days) for: xyz:RELIANCE.
Last run: 2026-11-05 09:14 EST.
Closed: xyz:NET long (07f2).
=== WHAT WENT WELL ===


Window,Update,Detail
Since last run,Biggest open-position gains,xyz:AAPL (c248) $2.62
Last 24h,Closed at a profit,"$26.64: MELANIA $17.33, INIT $9.31"
Last 24h,Funding received,$3.20
Now,Stop coverage,All 22 positions fully covered


=== NEEDS ATTENTION ===


Window,Update,Detail
Now,Aggregate risk over cap,$860.35 used vs $587.96 cap (-$272.38 room)
Since last run,PnL including open positions,-$12.49 (-0.33%)
Since last run,Closed at a loss,-$6.61: xyz:NET -$6.61
Since last run,Biggest open-position losses,"GOAT (07f2) -$5.25, INJ (07f2) -$3.81, ETH (07f2) -$3.56"
Last 24h,PnL including open positions,-$97.51 (-2.53%)
Last 24h,Closed at a loss,"-$142.86: xyz:SHAZ -$43.57, xyz:GOOGL -$27.98, INJ -$19.44, XPL -$14.28, USUAL -$7.06, 9 more"
Now,Available margin low,"$83.16 left of $2,351.85 above the reserve"
Now,Within 25% of liquidation,xyz:RELIANCE 24.8%


## Account / PnL

Combined account and risk summary. 24H rows are rolling windows ending now and exclude deposits and withdrawals; daily statistics and calendar-period cutoffs use US Eastern days.


In [4]:
analytics.show_account_pnl()


=== ACCOUNT / PNL AND RISK ===


Metric,Value
Account value,"$3,754.15"
Account 24H,-2.45%
Realized 24H,$-113.70
Realized 24H (% of account value),-3.03%
Unrealized PnL,$2.48
Unrealized PnL (% of account value),0.07%
Unrealized 24H change,$32.43
Average realized PnL / day,$-0.33
Average unrealized PnL change / day,n/a
--- RISK ---,


## Account charts

Account value and PnL charts. Before all wallets have saved history, account value and weekly returns use the oldest wallet alone; from shared coverage onward they use the combined accounts.


In [5]:
analytics.show_account_charts()


Weekly return chart omits 55 inactive weeks; statistics still include them.


## Account vs BTC

The daily account-vs-BTC comparison uses a time-weighted return across all wallets, so deposits, withdrawals, and transfers do not move it.


In [6]:
analytics.show_account_vs_btc()


The account line is a time-weighted return: deposits, withdrawals, and transfers do not move it. BTC is shown on the same dates.


## Traded volume vs realized PnL chart

Daily fill volume and realized PnL pooled across wallets.


In [7]:
analytics.show_volume_chart()


## Open positions

Show each wallet position separately, followed by an aggregate summary. The summary includes cumulative open-position funding as a percentage of combined account equity and total position value. Reserve-adjusted available margin is combined margin collateral (USDC on unified accounts) less the configured reserve; the summary shows that capacity, margin used, remaining available margin in dollars and as a percentage of that capacity, and position value as a percentage of equity. Set `MARGIN_RESERVE_PCT` (20% by default) in the setup cell to change the reserve. The position table includes funding and fees since open as dollar amounts, plus fees since open as a percentage of position value and position value as a percentage of wallet account value. Fee ratios are available for complete USDC-denominated fee history. Aggregate risk used and room apply `AGGREGATE_RISK_PCT` (20% of collateral by default) to `ATR_STOP_MULTIPLE` (1.25) x daily ATR over `ATR_PERIOD` (14 completed candles) plus an adverse five-day funding reserve; same-side positions across wallets are combined.

In [8]:
analytics.show_open_positions()


Wallet,Ticker,Side,Liq %,ROE,uPnL,Funding (since open),Funding (1h),Funding APR,Expected Price PnL (5d),Expected Funding PnL (5d),Expected PnL incl. funding (5d),Expected Price Return (5d),Expected Funding Return (5d)
07f2,INJ,SHORT,106.00%,-16.80%,-$46.64,$0.18,0.001250%,10.95%,-$7.02,$1.07,-$5.95,-0.607%,0.093%
07f2,ETH,SHORT,122.80%,6.51%,$17.80,$0.19,0.001250%,10.95%,$3.41,$1.00,$4.41,0.317%,0.093%
07f2,GOAT,SHORT,108.10%,5.70%,$20.70,$0.60,0.001250%,10.95%,$13.85,$3.44,$17.29,1.297%,0.322%
07f2,RUNE,SHORT,169.97%,0.20%,$0.36,$0.30,0.001250%,10.95%,$0.05,$1.08,$1.13,0.007%,0.150%
c248,XPL,SHORT,127.70%,-15.85%,-$21.85,$0.32,0.001250%,10.95%,$6.35,$1.23,$7.58,1.109%,0.214%
07f2,TRX,SHORT,271.68%,-16.26%,-$7.55,$0.16,-0.001912%,-16.75%,-$0.73,-$1.44,-$2.17,-0.154%,-0.305%
07f2,PONS,SHORT,244.99%,4.50%,$7.19,$0.09,0.001250%,10.95%,$20.49,$1.02,$21.51,4.349%,0.216%
c248,NEAR,SHORT,169.55%,12.22%,$13.63,$0.09,0.001250%,10.95%,-$6.37,$0.51,-$5.86,-1.477%,0.119%
c248,USUAL,SHORT,162.26%,-3.45%,-$4.60,$1.63,0.001250%,10.95%,$4.45,$1.00,$5.44,1.097%,0.246%
07f2,SAGA,SHORT,360.95%,7.42%,$8.11,$0.48,0.006867%,60.15%,$2.34,$0.61,$2.95,0.731%,0.191%


Wallet,Ticker,Side,Entry,Mark,Pos,Margin,Margin Type,Margin %,Fees,Fees / Position (%),Position / Account (%),Liq Px,Lev.,Sz
07f2,INJ,SHORT,7.9375,8.2625,"$1,157",$289,Cross,19.08%,-$0.16,0.014%,41.72%,17.021,4.00x,-139.90
07f2,ETH,SHORT,"2,426.6","2,386.4","$1,076",$269,Cross,17.73%,-$0.16,0.015%,38.78%,"5,316.9",4.00x,-0.45
07f2,GOAT,SHORT,0.014101,0.013827,"$1,069",$356,Cross,23.49%,-$0.16,0.015%,38.53%,0.028774,3.00x,"-77,249.00"
07f2,RUNE,SHORT,0.63145,0.63109,$720,$180,Cross,11.88%,-$0.32,0.044%,25.97%,1.7037,4.00x,"-1,141.30"
c248,XPL,SHORT,0.061984,0.06439,$573,$143,Cross,19.34%,-$0.24,0.043%,58.45%,0.14662,4.00x,"-8,895.00"
07f2,TRX,SHORT,0.313,0.31809,$472,$47,Cross,3.11%,-$0.20,0.043%,17.02%,1.1823,10.00x,"-1,484.00"
07f2,PONS,SHORT,0.22184,0.21837,$472,$157,Cross,10.37%,-$0.07,0.015%,17.00%,0.75335,3.00x,"-2,158.00"
c248,NEAR,SHORT,4.0846,3.9524,$432,$108,Cross,14.59%,-$0.06,0.015%,44.10%,10.654,4.00x,-109.20
c248,USUAL,SHORT,0.01005,0.01018,$405,$135,Cross,18.21%,-$0.18,0.043%,41.28%,0.026698,3.00x,"-39,803.50"
07f2,SAGA,SHORT,0.01739,0.01696,$320,$107,Cross,7.03%,-$0.05,0.015%,11.53%,0.078177,3.00x,"-18,854.00"


Wallet,Ticker,Side,Liq %,ROE,uPnL,Funding (since open),Funding (1h),Funding APR,Expected Price PnL (5d),Expected Funding PnL (5d),Expected PnL incl. funding (5d),Expected Price Return (5d),Expected Funding Return (5d)
c248,xyz:AAPL,LONG,n/a,3.73%,$8.46,$0.11,-0.002448%,-21.44%,$2.45,-$0.00,$2.45,0.356%,-0.000%
07f2,xyz:RELIANCE,SHORT,24.79%,-5.39%,-$1.02,$0.07,-0.026342%,-230.75%,-$4.98,$3.37,-$1.61,-8.622%,5.832%
,Net/total,,,3.00%,$7.44,$0.19,,,-$2.53,$3.37,$0.84,-0.339%,0.451%


Wallet,Ticker,Side,Entry,Mark,Pos,Margin,Margin Type,Margin %,Fees,Fees / Position (%),Position / Account (%),Liq Px,Lev.,Sz
c248,xyz:AAPL,LONG,349.88,354.08,$689,$230,Cross,30.98%,-$0.02,0.003%,70.23%,0,3.00x,1.94
07f2,xyz:RELIANCE,SHORT,13.52214,13.764,$58,$18,Isolated,1.18%,-$0.00,0.009%,2.08%,17.176,3.22x,-4.20
,Net/total,,,,$746,$247,,,-$0.02,0.003%,19.88%,,3.02x,


All positions: notional $8,241 | margin $2,269 | uPnL $2.48.
Funding and fees are cumulative since the position was opened. Fee ratios are shown only for complete USDC-denominated fees; otherwise they are n/a.
=== ACCOUNT NET EXPOSURE ===


Net Direction,Long/Short Positions,Long/Short Notional ($),Gross Notional ($),Net Notional ($),Net Exposure / Gross (%)
NET SHORT,2/20,"$707.84/$7,532.75","$8,240.59","$6,824.91",82.82%


... (3 outputs left out)

Coin,Side,Size,Position Value ($),Current Funding / 1h (%),Funding APR (%),Estimated Funding PnL / hour ($),Estimated Funding PnL / day ($),Funding Coverage (%)
AERO,LONG,30,$19.21,0.001250%,10.95%,$-0.0002,$-0.01,100.0%
ETH,SHORT,0.4506,"$1,075.63",0.001250%,10.95%,$0.0134,$0.32,100.0%
GOAT,SHORT,7.725e+04,"$1,068.59",0.001250%,10.95%,$0.0134,$0.32,100.0%
GRASS,SHORT,201,$111.74,0.001250%,10.95%,$0.0014,$0.03,100.0%
HEMI,SHORT,"4,096",$18.33,0.001250%,10.95%,$0.0002,$0.01,100.0%
IMX,SHORT,"1,467",$178.45,0.001250%,10.95%,$0.0022,$0.05,100.0%
INIT,SHORT,424,$57.91,0.001250%,10.95%,$0.0007,$0.02,100.0%
INJ,SHORT,139.9,"$1,157.10",0.001250%,10.95%,$0.0145,$0.35,100.0%
NEAR,SHORT,109.2,$432.41,0.001250%,10.95%,$0.0054,$0.13,100.0%
PENGU,SHORT,2.655e+04,$188.68,0.001250%,10.95%,$0.0024,$0.06,100.0%


Funding rates are current hourly rates, weighted by position notional. Positive rates are paid by longs and received by shorts; estimated cash flows assume the current rate stays constant. APR is simple annualization, not a forecast. The Net/total rate and APR are net cash flow on covered notional: positive means received.
Limited daily history; ATR uses all completed candles available (shorter than 14 days) for: xyz:RELIANCE.


Metric,Value
Wallets,2
Position value,"$8,241"
uPnL,$2.48
ROE,0.11%
Margin,"$2,269"
Margin % bal,60.43%
Available margin (80% of collateral),"$2,351.85"
Available margin used (20% reserve),96.46%
Available margin remaining,$83.16
Available margin remaining (%),3.54%


ROE is unrealized PnL divided by used margin; a negative ROE means the open positions are currently at an unrealized loss relative to the margin supporting them.


## Net funding APR over time

Applies the open-positions `Net Funding APR (%) (simple, covered)` formula to Hyperliquid funding history: net funding cash flow divided by gross notional, annualized, pooled across wallets. The history records actual funding on positions held at the time; the summary table shows the current rate on current positions, so the two differ. Daily values use US Eastern days, and the 7-day line is notional-weighted. Positive means funding received.


In [9]:
_ = analytics.show_funding_apr_chart()


Latest 7-day net funding APR: 25.69% (2026-11-05, America/New_York days). Lifetime: -6.39% on $-50.79 net funding.


## Realized PnL statistics

Pool spot/perp fills and signed funding across wallets.


In [10]:
analytics.show_realized_pnl()


Metric,Value
Realized PnL,$-199.45 (-5.31%)
24H,$-113.70 (-3.03%)
7D,$155.05 (4.13%)
30D,$363.14 (9.67%)
QTD,$370.14 (9.86%)
YTD,$369.94 (9.85%)
W,236
Win %,62.93%
Avg W,$7.89
L,139


Metric,Value
Realized PnL,$22.01 (0.59%)
24H,$0.00 (0.00%)
7D,$0.00 (0.00%)
30D,$12.16 (0.32%)
QTD,$22.01 (0.59%)
YTD,$22.01 (0.59%)
W,3
Win %,100.00%
Avg W,$7.34
L,0



=== PERP PNL BREAKDOWN ===


Component,Amount ($),Contribution (% of |net PnL|)
Trading,$-37.54,-18.82%
Fees,$-111.11,-55.71%
Funding,$-50.79,-25.47%
Net perp PnL,$-199.45,


Liquidations: 0 of 375 closes (0.00%) | liquidation PnL: $0.00.


## Traded volume vs PnL

Combined turnover efficiency and correlations.


In [11]:
analytics.show_volume_summary()


=== TRADED VOLUME VS REALIZED PNL (ALL WALLETS) ===


Market,Volume ($),Realized PnL excl. funding (trading + fees) ($),Realized PnL / Volume (%),PnL / $1k volume ($),Active days,Daily Pearson r,Daily Spearman r
Perps,"$280,143.03",$-148.66,-0.05%,$-0.53,61,0.061,0.195
Spot,"$1,397.53",$22.01,1.58%,$15.75,5,-0.511,-0.462
Net/total,"$281,540.56",$-126.64,-0.04%,$-0.45,61,,


## Execution quality

Last 30 days of perp fills split into opening and closing, maker and taker, with the fees paid. Stop-losses must stay taker, so closing taker fills are only partly avoidable.

In [12]:
analytics.show_execution_quality()

Fills,Type,Count,Notional ($),Share (%),Fees ($),Fees (bps)
Opening,Maker,133,"$22,302.90",12.5%,$2.74,1.23
Opening,Taker,300,"$63,309.09",35.6%,$25.93,4.10
Closing,Maker,123,"$11,179.81",6.3%,$1.13,1.01
Closing,Taker,180,"$81,167.67",45.6%,$35.62,4.39
Net/total,,736,"$177,959.47",100.0%,$65.42,3.68


Taking liquidity cost about $44.86 more in fees than the maker rate would have. This ignores missed fills and worse prices from waiting, and stop-losses must stay taker.


## PnL attribution

Combined realized and open-position PnL by asset.


In [13]:
analytics.show_pnl_attribution()


Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
BTC,12,$-494.29,-19.25%,$0.00,0.00%,$-494.29,-17.89%
kBONK,3,$-398.21,-15.51%,$0.00,0.00%,$-398.21,-14.41%
ZORA,21,$299.47,11.66%,$0.00,0.00%,$299.47,10.84%
KAITO,4,$-194.09,-7.56%,$0.00,0.00%,$-194.09,-7.03%
HYPE,11,$140.87,5.49%,$0.00,0.00%,$140.87,5.10%
ETH,15,$99.23,3.86%,$17.80,8.97%,$117.03,4.24%
TRUMP,8,$107.90,4.20%,$0.00,0.00%,$107.90,3.91%
VVV,40,$77.41,3.01%,$0.00,0.00%,$77.41,2.80%
INIT,9,$54.28,2.11%,$12.15,6.12%,$66.43,2.40%
INJ,4,$-14.77,-0.58%,$-46.64,-23.50%,$-61.41,-2.22%


Asset,Closes,Realized,Realized (% gross),uPnL,uPnL (% gross),Total PnL,Total (% gross)
xyz:GOOGL,3,$-105.17,-30.34%,$0.00,0.00%,$-105.17,-30.01%
io:ANTH,2,$-86.80,-25.05%,$0.00,0.00%,$-86.80,-24.77%
xyz:SHAZ,4,$-25.97,-7.49%,$0.00,0.00%,$-25.97,-7.41%
xyz:ZHIPU,1,$24.64,7.11%,$0.00,0.00%,$24.64,7.03%
xyz:NET,5,$-24.33,-7.02%,$0.00,0.00%,$-24.33,-6.94%
xyz:JIO,24,$14.73,4.25%,$0.00,0.00%,$14.73,4.20%
xyz:ORCL,1,$10.45,3.02%,$0.00,0.00%,$10.45,2.98%
para:SOFI,1,$-10.12,-2.92%,$0.00,0.00%,$-10.12,-2.89%
xyz:LITE,1,$9.44,2.72%,$0.00,0.00%,$9.44,2.69%
io:GPRO,4,$8.78,2.53%,$0.00,0.00%,$8.78,2.51%


Asset,Closes,Realized,Realized (% gross)
"Other (vaults, transfers, open spot)",0,$31.18,58.61%
HYPE,3,$22.01,41.39%
Net/total,3,$53.19,


Net attributed PnL across wallets: $-143.78. The Spot table's Other row is Hyperliquid's non-perp PnL less spot fill PnL, so the net matches Hyperliquid's total to within small timing differences.
